In [1]:
import pandas as pd
import numpy as np
import sumolib
from modules.sumo import TLIncomingCorridors

In [2]:
all_routes_df=None

route_files_list=["./Ingolstadt Network/InTAS_016.rou.xml","./Ingolstadt Network/InTAS_017.rou.xml","./Ingolstadt Network/InTAS_018.rou.xml"]
for route_file in route_files_list:
    route_df=pd.read_xml(route_file,iterparse={"route":["edges"]})
    route_df=route_df.rename(columns={"edges":"route_edges"})
    if all_routes_df is None:
        all_routes_df=route_df
    else:
        all_routes_df=pd.concat([all_routes_df,route_df],axis=0,ignore_index=True)

In [18]:
def extract_tl_routes(full_route,net):
    full_route=full_route.split(" ")
    tl_routes={}
    start_balise=0
    end_balise=0
    for i,edge_id in enumerate(full_route):
        edge=net.getEdge(edge_id)
        to_node=edge.getToNode()
        to_node_type=to_node.getType()
        if to_node_type== "traffic_light":
            connections=to_node.getConnections()
            for connection in connections:
                tl=connection.getTLSID()
                if tl:
                    break
            if tl:
                end_balise=i+1
                if tl_routes.get(tl) is None:
                    tl_routes.setdefault(tl, [])
                tl_routes[tl].append(full_route[start_balise:end_balise])
                start_balise=end_balise
    return tl_routes

net=sumolib.net.readNet("./Ingolstadt Network/ingolstadt.net.xml",withInternal=False)
all_routes_df["tl_routes"]=all_routes_df["route_edges"].apply(extract_tl_routes,args=(net,))

print(all_routes_df["tl_routes"].values)

[{'2033094557': [['128668757', '439478423', '-439478423', '-202089410#1', '-198768615#0', '-202089408#1', '-202089408#0', '-202089408#0.71', '-192768909#1']], '26423900': [['gneE150', 'gneE150.157', 'gneE153']], '26423915': [['25508272#0', '25508272#1', '614568192', '126909938', '27267723']], '89173763': [['8033457#0', '42738709#0', '203127018', '137672872', '137672874#0', '320733913', '320733915', '77414799#0', '77414803#0', '136888519#0', '136888519#1', '80518321', '32062034#0', '32062034#2', '32062034#4', '18809238#0', '18809238#1', '18809238#2', '201201938#0', '201201938#1', '201201938#3', '393244241#0', '393244241#1', '35744154#0', '378946906', '449057059', '449057052', '378946907', '379795373', '449057060', '148527692#0', '148527692#1', '128361102#0', '285549250#0']]}
 {'cluster_306484187_cluster_1200363791_1200363826_1200363834_1200363898_1200363927_1200363938_1200363947_1200364074_1200364103_1507566554_1507566556_255882157_306484190': [['201089423#0', '201089423#2', '32124744',

In [ ]:
net_xml="./Ingolstadt Network/ingolstadt.net.xml"
route_files_list=["./Ingolstadt Network/InTAS_016.rou.xml","./Ingolstadt Network/InTAS_017.rou.xml","./Ingolstadt Network/InTAS_018.rou.xml"]

corridors_generator=TLIncomingCorridors(net_file=net_xml,route_files_list=route_files_list)
corridors_df=corridors_generator.get_corridors_df()

(12212, 1)
CORRIDORS EXTRACTED
CORRIDORS DF GENERATED
